# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam12arshad17/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

print("Connected! Ready to query.")

Connected! Ready to query.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content page (`content_hash_id`) for one
client (`client_hash_id`), on one calendar day (`report_date`).

**Table:** `fact_content_daily_performance`, partitioned by `month=YYYY-MM`.

**Time window:**
- Working month: `month=2026-03` — a mid-panel month, used to build and test all
  logic in this notebook.
- Sealed test month: `month=2026-06` — the final month in the panel. It is treated
  as untouched, held-out data and is never used to develop label logic.

In [18]:
con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (safe model inputs — known before the outcome):
- `gsc_impressions` — trailing search visibility, observed before any future outcome.
- `gsc_clicks` — trailing click volume from the same past window.
- `gsc_avg_position` — average ranking position observed in the past window.
- `ga4_sessions` — trailing analytics traffic, same past-only logic.
- `gsc_data_available` — flags whether GSC data exists for this row; used to filter.

**Label / Proxy:**
- `is_declining` (built below) — whether a content item's impressions dropped in the
  second half of the month vs. the first half. Computed *after* the observation
  window, never used as a feature.

**Context** (used for grouping/joining, not as model inputs):
- `client_hash_id`, `content_hash_id`, `report_date`

**Excluded:**
- `month` — only used to select the analysis window (2026-03); not predictive, so
  it's dropped before modeling.
- `client_has_gsc` / `client_has_ga4` — describe data-source access at the client
  level, not the content's actual performance; excluded to avoid mixing access
  flags with performance signal.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries confirm the contract claims above:

1. **Grain check** — confirms one row per (report_date, client_hash_id, content_hash_id).
2. **Row count + date span** — confirms the March 2026 slice size and date range.
3. **Availability** — filters with `IS TRUE` to show how many rows have usable GSC data.

In [20]:
# Query 1 — grain check
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
print("Duplicate groups found:", len(grain_check))
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate groups found: 0


,report_date,client_hash_id,content_hash_id,n


In [21]:
# Query 2 — row count and date span
row_count_span = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS start_date, MAX(report_date) AS end_date
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
row_count_span

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [22]:
# Query 3 — availability, filtered with IS TRUE
availability = con.sql(f"""
    SELECT COUNT(*) AS available_rows
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


Five features, built from March 2026 data, each knowable before the outcome window:

1. **gsc_impressions_total** — trailing search visibility, summed over the month;
   observed before any future outcome.
2. **gsc_clicks_total** — trailing click volume, same past-only window.
3. **gsc_avg_position** — average ranking position observed in the past window only.
4. **ctr** — clicks/impressions ratio, computed purely from past numbers.
5. **days_observed** — number of days this content had data in March; a fixed
   historical fact, independent of any future outcome.

In [23]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_total,
        SUM(gsc_clicks) AS gsc_clicks_total,
        AVG(gsc_avg_position) AS gsc_avg_position,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE 0 END AS ctr,
        COUNT(*) AS days_observed
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print("Feature frame shape:", features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 7)


,client_hash_id,content_hash_id,gsc_impressions_total,gsc_clicks_total,gsc_avg_position,ctr,days_observed
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.145765,0.001112,31
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,4.909314,0.000000,17
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,0.0,5.177774,0.000000,31
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,772.0,1.0,4.685335,0.001295,31
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,14.0,0.0,4.266667,0.000000,10


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Build the label: did impressions drop in the second half of March vs the first half?
label_data = con.sql(f"""
    WITH halves AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        client_hash_id,
        content_hash_id,
        (second_half < first_half) AS is_declining,
        (second_half - first_half) AS impression_change   -- this IS the leak
    FROM halves
""").df()

df = features.merge(label_data, on=["client_hash_id", "content_hash_id"])

X_cols = ["gsc_impressions_total", "gsc_clicks_total", "gsc_avg_position", "ctr", "days_observed"]

# --- HONEST score: only the 5 real features ---
X_train, X_test, y_train, y_test = train_test_split(
    df[X_cols], df["is_declining"], test_size=0.3, random_state=0
)
honest_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])
print("Honest AUC (5 real features):", round(honest_auc, 3))

# --- LEAKED score: sneak in impression_change, the exact thing the label came from ---
X_leak_cols = X_cols + ["impression_change"]
X_train, X_test, y_train, y_test = train_test_split(
    df[X_leak_cols], df["is_declining"], test_size=0.3, random_state=0
)
leaked_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
leaked_auc = roc_auc_score(y_test, leaked_model.predict_proba(X_test)[:, 1])
print("Leaked AUC (with impression_change) — watch it jump toward 1.0:", round(leaked_auc, 3))

# --- Remove the leak, keep the honest number ---
df = df.drop(columns=["impression_change"])
print(f"\nLesson: honest AUC = {honest_auc:.3f} vs leaked AUC = {leaked_auc:.3f}.")
print("impression_change is discarded — it IS the quantity the label was derived from.")

Honest AUC (5 real features): 0.572
Leaked AUC (with impression_change) — watch it jump toward 1.0: 1.0

Lesson: honest AUC = 0.572 vs leaked AUC = 1.000.
impression_change is discarded — it IS the quantity the label was derived from.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has several limitations:

- **Unbalanced panel:** Clients have different amounts of historical data
  (`gsc_data_start` varies widely across `dim_clients`), so comparisons across
  clients aren't always apples-to-apples.
- **Partial availability:** Only ~37% of March rows have `gsc_data_available IS TRUE`
  (3.6M of 9.8M rows) — most rows can't be used for GSC-based analysis at all.
- **Arbitrary label boundary:** The `is_declining` label used here splits the month
  at March 16 — a reasonable but arbitrary midpoint. A different cutoff could
  reclassify borderline content items.
- **Correlational, not causal:** The data shows *what* happened to impressions and
  clicks, but not *why* — it can't explain business reasons behind a decline
  (algorithm updates, seasonality, competitor changes, etc.).

In [25]:
history_spread = con.sql("""
    SELECT
        MIN(gsc_data_start) AS earliest_client,
        MAX(gsc_data_start) AS latest_client
    FROM read_parquet('{}/dim_clients.parquet')
""".format(BASE)).df()
history_spread


,earliest_client,latest_client
0,2025-01-27,2026-06-02


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.